# 13F 美股 Top50 策略优化系统（方案 B - 模块化研究交付版）

本 Notebook 作为研究系统的轻量入口，遵循方案 B 模块化设计规范：
- **严格时间点（Point-in-Time）**：杜绝未来函数泄漏，标的池由重训日前已披露 13F 滚动构建。
- **真实会计与撮合**：考虑佣金、滑点、融资与借券成本，隔夜真实结转。
- **算法真实闭环**：DAgger 状态真实转移，IRL 奖励网络参与受约束优化并支持自动回退。
- **独立风控对冲**：Bottom-M 空头与多头 Softmax 彻底解耦。
- **M0—M7 严格样本外消融**：仅在显著提升风险调整收益时启用高级模块。

In [ ]:
# 单元 1：环境导入与包路径配置
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from top50_strategy.config import RunConfig
from top50_strategy.pipeline import run_research, SyntheticAdapters

print('top50_strategy 模块与研究环境就绪。')

In [ ]:
# 单元 2：加载冻结基准研究配置
config_path = PROJECT_ROOT / 'configs' / 'baseline.toml'
config = RunConfig.from_toml(config_path)

print(f'研究参数摘要:')
print(f'  - 滚动股票池: Top {config.universe_size} 标的 (回溯 {config.lookback_quarters} 个季度)')
print(f'  - 多头配置: Top-{config.top_k}，杠杆范围 [{config.panic_scale:.2f}x, {config.bull_leverage:.2f}x]')
print(f'  - 空头对冲: Bottom-{config.bottom_m} (独立动量/宏观门控触发)')
print(f'  - 摩擦成本: 佣金 {config.commission_rate*10000:.1f}bps, 滑点 {config.slippage_rate*10000:.1f}bps, 借券费率 {config.short_borrow_rate:.1%}')

In [ ]:
# 单元 3：执行端到端研究流水线与消融实验 (M0 - M7)
dates = pd.bdate_range('2018-01-01', '2024-12-31', tz='UTC')
top50_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'NVDA', 'META', 'TSLA', 'BRK.B', 'JPM', 'JNJ',
    'V', 'PG', 'UNH', 'HD', 'MA', 'BAC', 'DIS', 'ADBE', 'CRM', 'NFLX',
    'XOM', 'CVX', 'KO', 'PEP', 'ABT', 'MRK', 'PFE', 'TMO', 'COST', 'WMT',
    'MCD', 'CSCO', 'ACN', 'ABBV', 'LIN', 'VZ', 'NEE', 'DHR', 'PM', 'TXN',
    'AMD', 'QCOM', 'HON', 'INTC', 'UNP', 'LOW', 'SPGI', 'IBM', 'GE', 'CAT'
]

adapters = SyntheticAdapters(dates, top50_tickers)
output_dir = PROJECT_ROOT / 'artifacts'

print('正在运行端到端严谨点位回测与样本外消融实验...')
report = run_research(config, adapters.filing_adapter, adapters.market_adapter, output_dir=output_dir)
print('流水线运行完毕，实验结果已自动归档至 artifacts/ 目录。')

In [ ]:
# 单元 4：数据质量与时间点审计报告
audit = report.data_audit
print('==================== 数据质量审计报告 (Point-in-Time Audit) ====================')
print(f'  - 未来信息泄漏次数 (Future Access Violations) : {audit.future_access_count} (合格)')
print(f'  - 审计 13F 申报总数 (Total Filings Processed)   : {audit.total_filings}')
print(f'  - 有效持仓记录条数 (Total Records Parsed)       : {audit.total_records}')
print(f'  - 覆盖资产代码总数 (Unique Assets Covered)      : {audit.unique_tickers}')
print(f'  - 首次可用披露时间 (First Available Date)       : {audit.first_available_date}')
print('================================================================================')

In [ ]:
# 单元 5：展示 M0 — M7 样本外消融评估矩阵
print('================================ M0 - M7 策略样本外消融实验对比表 ================================')
display_df = report.ablation_table.copy()
print(display_df.to_string(index=False))
print('=================================================================================================')

In [ ]:
# 单元 6：高级算法模块准入与启用决策
print('============================= 高级算法启用门槛决策 (Gate Decisions) =============================')
for mod, dec in report.enablement_decisions.items():
    status = '【已启用】' if dec.enabled else '【默认关闭】'
    print(f'  > 模块 {mod:<3}: {status} - {dec.reason}')
print('=================================================================================================')

In [ ]:
# 单元 7：导出最新目标配置信号 (Latest Trading Signals)
print('======================= 最新实盘/离线调仓目标信号 (Latest Allocation) =======================')
sorted_weights = sorted(report.latest_weights.items(), key=lambda x: x[1], reverse=True)
for ticker, w in sorted_weights:
    print(f'  > 资产 {ticker:<6}: 目标多头权重 {w*100:6.2f}%')
print('=============================================================================================')